# 🛠️ Advanced Tooling & MCP for SOC Agents

**Goal:** Equip SOC agents with production-ready tools to interact directly with the security stack.

| Tool Type | Use Case | SDK Class |
|-----------|----------|----------|
| **FunctionTool** | Custom Python → SIEM/XDR API wrappers | `FunctionTool` (explicit JSON Schema) |
| **OpenAPI Tool** | Call REST APIs directly from agent spec | `OpenApiTool` |
| **MCP Tool** | Connect to Model Context Protocol servers | `MCPTool` |
| **Bing Grounding** | Real-time web threat intel | `BingGroundingTool` |

> 💡 MCP (Model Context Protocol) is how agents natively integrate with security platforms like Microsoft Sentinel, Defender XDR, and custom SOC tooling.
>
> **SDK:** `azure-ai-projects>=2.0.0` — Responses API with explicit function-call resolution loops.


In [ ]:
!uv pip install "azure-ai-projects>=2.0.0" azure-identity python-dotenv rich pandas matplotlib -q


In [ ]:
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

load_dotenv(Path('..') / '.env', override=False)

AZURE_AI_PROJECT_ENDPOINT = os.getenv('AZURE_AI_PROJECT_ENDPOINT', '')
MODEL_DEPLOYMENT_NAME = os.getenv('MODEL_DEPLOYMENT_NAME', 'gpt-4o')
MCP_SERVER_URL = os.getenv('MCP_SERVER_URL', '')
MCP_SERVER_LABEL = os.getenv('MCP_SERVER_LABEL', 'sentinel-mcp')

MOCK_MODE = not bool(AZURE_AI_PROJECT_ENDPOINT)
status = '⚠️  MOCK MODE' if MOCK_MODE else f'✅ Connected: {AZURE_AI_PROJECT_ENDPOINT[:50]}...'
print(status)


## 🔵 Part 1: FunctionTool — SIEM & XDR API Wrappers

FunctionTools let agents call any Python function. Here we wrap three common SOC integrations:
- Sentinel KQL query executor
- Defender XDR alert retrieval
- IP reputation enrichment

In [ ]:
# ═══ azure-ai-projects 2.x FunctionTool: explicit JSON Schema per tool ═══════
# No docstring-based auto-parsing — define name, description, parameters, strict=True

def run_sentinel_kql_query(kql_query: str, timespan_hours: int = 24) -> str:
    """Execute a KQL query against Microsoft Sentinel and return the results."""
    # Production: use azure-monitor-query SDK
    mock_results = {
        'query': kql_query[:100],
        'timespan_hours': timespan_hours,
        'row_count': 3,
        'rows': [
            {'TimeGenerated': '2025-03-07T02:10:00Z', 'IPAddress': '185.220.101.47', 'ResultType': '50126', 'UserPrincipalName': 'jsmith@corp.com'},
            {'TimeGenerated': '2025-03-07T02:13:00Z', 'IPAddress': '185.220.101.47', 'ResultType': '50126', 'UserPrincipalName': 'jsmith@corp.com'},
            {'TimeGenerated': '2025-03-07T02:14:33Z', 'IPAddress': '185.220.101.47', 'ResultType': '0',     'UserPrincipalName': 'jsmith@corp.com'},
        ]
    }
    return json.dumps(mock_results)


def get_defender_xdr_alerts(severity: str, hours_back: int = 24) -> str:
    """Retrieve active Microsoft Defender XDR alerts filtered by severity."""
    # Production: use Microsoft Graph Security API
    mock_alerts = [
        {'id': 'da637935-7e5e-4c35-9b66-a9c5de7d4eb8', 'title': 'Cobalt Strike C2 Communication', 'severity': severity, 'status': 'Active', 'machineId': 'WS-DEV-04'},
        {'id': 'da637935-7e5e-4c35-9b66-b8c5de7d4ec9', 'title': 'Suspicious PowerShell Execution', 'severity': severity, 'status': 'Active', 'machineId': 'WS-FINANCE-04'},
    ] if severity in ('High', 'Critical') else []
    return json.dumps({'severity_filter': severity, 'hours_back': hours_back, 'alerts': mock_alerts, 'count': len(mock_alerts)})


def enrich_ip_reputation(ip_address: str) -> str:
    """Enrich an IP address with threat reputation data from multiple intel sources."""
    # Production: call VirusTotal API / Shodan / RiskIQ
    known_bad = {
        '185.220.101.47': {'threat_score': 9.5, 'category': 'TOR Exit Node', 'country': 'DE', 'asn': 'AS201814 MEVSPACE', 'vt_detections': '42/87', 'is_tor': True},
        '194.165.16.101': {'threat_score': 9.8, 'category': 'C2 Server', 'country': 'RU', 'asn': 'AS57044 HOSTKEY', 'vt_detections': '61/87', 'is_tor': False},
    }
    data = known_bad.get(ip_address, {'threat_score': 1.0, 'category': 'Unknown', 'country': 'N/A', 'asn': 'N/A', 'vt_detections': '0/87', 'is_tor': False})
    return json.dumps({'ip': ip_address, **data})


# ── FunctionTool schema objects ───────────────────────────────────────────────
from azure.ai.projects.models import FunctionTool

tool_kql = FunctionTool(
    name='run_sentinel_kql_query',
    description='Execute a KQL query against Microsoft Sentinel and return matching rows.',
    parameters={
        'type': 'object',
        'properties': {
            'kql_query':      {'type': 'string',  'description': 'KQL query string to execute against Sentinel.'},
            'timespan_hours': {'type': 'integer', 'description': 'Hours to query back. Default 24.'},
        },
        'required': ['kql_query'],
        'additionalProperties': False,
    },
    strict=True,
)

tool_defender = FunctionTool(
    name='get_defender_xdr_alerts',
    description='Retrieve active Microsoft Defender XDR alerts filtered by severity level.',
    parameters={
        'type': 'object',
        'properties': {
            'severity':   {'type': 'string',  'description': 'Severity filter: High, Medium, Low, or Informational.'},
            'hours_back': {'type': 'integer', 'description': 'Hours to look back. Default 24.'},
        },
        'required': ['severity'],
        'additionalProperties': False,
    },
    strict=True,
)

tool_enrichment = FunctionTool(
    name='enrich_ip_reputation',
    description='Enrich an IP address with threat reputation data — threat score, geolocation, ASN, and VirusTotal detections.',
    parameters={
        'type': 'object',
        'properties': {
            'ip_address': {'type': 'string', 'description': 'IPv4 or IPv6 address to enrich.'},
        },
        'required': ['ip_address'],
        'additionalProperties': False,
    },
    strict=True,
)

SOC_TOOL_FUNCTIONS = {
    'run_sentinel_kql_query':  run_sentinel_kql_query,
    'get_defender_xdr_alerts': get_defender_xdr_alerts,
    'enrich_ip_reputation':    enrich_ip_reputation,
}
print(f'✅ {len(SOC_TOOL_FUNCTIONS)} SOC tool functions + FunctionTool schemas ready')


In [ ]:
# 🔵 Test the tools locally BEFORE giving them to an agent
print('Testing KQL tool:')
result = json.loads(run_sentinel_kql_query('SigninLogs | where ResultType != 0 | take 5', timespan_hours=48))
df_kql = pd.DataFrame(result['rows'])
display(df_kql.style.set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'}))

print('\nTesting IP enrichment tool:')
for ip in ['185.220.101.47', '194.165.16.101', '8.8.8.8']:
    r = json.loads(enrich_ip_reputation(ip))
    icon = '🔴' if r['threat_score'] > 7 else '🟡' if r['threat_score'] > 3 else '🟢'
    print(f'  {icon} {ip} → Score: {r["threat_score"]} | {r["category"]} | {r["country"]}')

## 🔵 Visualize the Tool Ecosystem

Before building agents, let's map which tools address which SOC capabilities.

In [ ]:
tools_data = {
    'Tool': ['run_sentinel_kql_query', 'get_defender_xdr_alerts', 'enrich_ip_reputation', 'MCPTool (Sentinel)', 'MCPTool (Defender)', 'OpenApiTool (TI Feed)', 'BingGroundingTool'],
    'Type': ['FunctionTool', 'FunctionTool', 'FunctionTool', 'MCPTool', 'MCPTool', 'OpenApiTool', 'Bing'],
    'Category': ['SIEM', 'XDR', 'Threat Intel', 'SIEM', 'XDR', 'Threat Intel', 'OSINT'],
    'Auth': ['Managed Identity', 'Managed Identity', 'API Key', 'MCP Token', 'MCP Token', 'API Key', 'Connection'],
    'Latency_ms': [800, 600, 400, 300, 350, 500, 1200],
}

df_tools = pd.DataFrame(tools_data)
display(df_tools.style
        .map(lambda v: 'color: #4ecdc4' if v == 'FunctionTool' else 'color: #ff6b6b' if v == 'MCPTool' else 'color: #45b7d1', subset=['Type'])
        .set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'})
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#1e2127')

# Category distribution
cat_counts = df_tools['Category'].value_counts()
colors_cat = ['#4ecdc4', '#45b7d1', '#ff6b6b', '#ffdd44']
ax1.bar(cat_counts.index, cat_counts.values, color=colors_cat[:len(cat_counts)], edgecolor='#444')
ax1.set_facecolor('#1e2127')
ax1.set_title('Tools by SOC Category', color='white')
ax1.tick_params(colors='white')
for s in ax1.spines.values(): s.set_color('#444')

# Latency comparison
type_colors = {'FunctionTool': '#4ecdc4', 'MCPTool': '#ff6b6b', 'OpenApiTool': '#45b7d1', 'Bing': '#ffdd44'}
bar_colors = [type_colors.get(t, '#888') for t in df_tools['Type']]
ax2.barh(df_tools['Tool'], df_tools['Latency_ms'], color=bar_colors, edgecolor='#444', height=0.6)
ax2.set_facecolor('#1e2127')
ax2.set_title('Estimated Tool Latency (ms)', color='white')
ax2.tick_params(colors='white')
ax2.set_xlabel('Latency (ms)', color='white')
for s in ax2.spines.values(): s.set_color('#444')

handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
ax2.legend(handles=handles, facecolor='#1e2127', labelcolor='white', fontsize=8)

plt.suptitle('SOC Agent Tool Ecosystem', color='white', fontsize=13)
plt.tight_layout()
plt.show()


## 🔴 Create an Agent with FunctionTools

In [ ]:
if not MOCK_MODE:
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import PromptAgentDefinition
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential()
    project_client = AIProjectClient(endpoint=AZURE_AI_PROJECT_ENDPOINT, credential=credential)
    openai_client = project_client.get_openai_client()

    soc_analyst = project_client.agents.create_version(
        agent_name='soc-analyst-with-tools',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a SOC Level-2 analyst with direct access to Sentinel (KQL) and Defender XDR. '
                'When investigating an IP or alert, always: '
                '1) Enrich the IP with reputation data. '
                '2) Query Sentinel for related sign-in events. '
                '3) Check Defender for active XDR alerts. '
                'Summarize findings in a structured investigation note.'
            ),
            tools=[tool_enrichment, tool_kql, tool_defender],
        ),
    )
    print(f'✅ Agent with {len(SOC_TOOL_FUNCTIONS)} function tools created: {soc_analyst.name}')
else:
    print('⚠️  Skipping — set AZURE_AI_PROJECT_ENDPOINT')


In [ ]:
if not MOCK_MODE:
    from openai.types.responses.response_input_param import FunctionCallOutput, ResponseInputParam

    def resolve_function_calls(response, fn_map: dict, agent_name: str, max_rounds: int = 8):
        """Resolve all function_call items returned by the Responses API."""
        for _ in range(max_rounds):
            calls = [item for item in response.output if item.type == 'function_call']
            if not calls:
                break
            inputs: ResponseInputParam = [
                FunctionCallOutput(
                    type='function_call_output',
                    call_id=fc.call_id,
                    output=(
                        json.dumps(fn_map[fc.name](**json.loads(fc.arguments)))
                        if fc.name in fn_map
                        else '{"error": "unknown function"}'
                    ),
                )
                for fc in calls
            ]
            response = openai_client.responses.create(
                input=inputs,
                previous_response_id=response.id,
                extra_body={'agent_reference': {'name': agent_name, 'type': 'agent_reference'}},
            )
        return response

    response = openai_client.responses.create(
        input='Investigate the suspicious IP 185.220.101.47 that just authenticated to our Azure portal. Check for related alerts and Sentinel activity.',
        extra_body={'agent_reference': {'name': soc_analyst.name, 'type': 'agent_reference'}},
    )
    response = resolve_function_calls(response, SOC_TOOL_FUNCTIONS, soc_analyst.name)

    print('Run status: complete')
    print('\n🤖 ANALYST REPORT:')
    print('─' * 60)
    print(response.output_text)
else:
    print('⚠️  Skipping run')


## 🔴 Part 2: MCPTool — Connect to Sentinel / Defender MCP Server

MCP (Model Context Protocol) is the **next evolution** beyond function tools — a standardized protocol that connects agents to any MCP-compatible security platform.

Microsoft offers MCP servers for:
- **Microsoft Sentinel** — KQL across workspaces, watchlists, analytics rules
- **Microsoft Defender XDR** — Advanced Hunting, Endpoint isolation, Alert management
- **Microsoft Entra** — Identity risk, conditional access

In `azure-ai-projects>=2.0.0`, the class is **`MCPTool`** (replaces old `McpTool`).
After `responses.create()`, check `response.output` for `mcp_approval_request` items and respond with `McpApprovalResponse`.

> ⚙️ Requires an MCP server endpoint — set `MCP_SERVER_URL` in your `.env`


In [ ]:
if not MOCK_MODE and MCP_SERVER_URL:
    from azure.ai.projects.models import MCPTool

    # In 2.x: MCPTool replaces McpTool — require_approval controls the human-in-the-loop gate
    mcp_tool = MCPTool(
        server_label=MCP_SERVER_LABEL,
        server_url=MCP_SERVER_URL,
        require_approval='always',  # For SOC: always require explicit approval for tool execution
    )

    mcp_agent = project_client.agents.create_version(
        agent_name='soc-mcp-sentinel-agent',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a SOC analyst with direct access to Microsoft Sentinel via MCP. '
                'Use MCP tools to run KQL queries, retrieve incidents, check analytics rules, '
                'and investigate threats in real time.'
            ),
            tools=[mcp_tool],
        ),
    )
    print(f'✅ MCP Agent created: {mcp_agent.name}')
    print(f'   MCP server: {MCP_SERVER_URL}')
    print(f'   Approval mode: always (human-in-the-loop for every tool call)')
else:
    print('⚠️  MCP_SERVER_URL not configured.')
    print('   To use MCP: deploy a Sentinel MCP gateway and set MCP_SERVER_URL in .env')
    print('   See: https://aka.ms/model-context-protocol')


In [ ]:
# 🔵 Understand MCP approval flow — runs with or without Azure credentials

mcp_flow = {
    'Step': ['Agent sends request', 'Agent selects MCP tool', 'Server returns mcp_approval_request', 'Client inspects + approves', 'MCP server executes', 'Agent receives result', 'Agent responds'],
    'Status': ['in_progress', 'in_progress', 'requires_action', 'approval_submitted', 'in_progress', 'in_progress', 'completed'],
    'Actor': ['SDK', 'LLM', 'Foundry Service', 'Your Code', 'MCP Server', 'Foundry Service', 'LLM'],
    'Security_Note': ['—', '—', 'Approval gate', '✅ Explicit consent', 'Sandboxed execution', '—', '—'],
}

df_mcp = pd.DataFrame(mcp_flow)
display(df_mcp.style
        .map(lambda v: 'color: #ff6b6b; font-weight: bold' if v == 'requires_action' else
                       'color: #4ecdc4' if v == 'completed' else '', subset=['Status'])
        .map(lambda v: 'color: #98d9a8' if '✅' in str(v) else '', subset=['Security_Note'])
        .set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'})
)

print()
print('💡 Approval loop pattern (azure-ai-projects 2.x):')
print('''
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

response = openai_client.responses.create(
    input="query sentinel...",
    extra_body={"agent_reference": {"name": mcp_agent.name, "type": "agent_reference"}},
)
inputs: ResponseInputParam = []
for item in response.output:
    if item.type == "mcp_approval_request":
        inputs.append(McpApprovalResponse(
            type="mcp_approval_response",
            approve=True,          # ← human decision here
            approval_request_id=item.id,
        ))
if inputs:
    response = openai_client.responses.create(
        input=inputs, previous_response_id=response.id,
        extra_body={"agent_reference": {...}},
    )
''')
print('   require_approval="always"  → human approves each call (write operations)')
print('   require_approval="never"   → auto-approve (safe for read-only queries only)')


## 🔵 Part 3: OpenAPI Tool — REST API as Agent Capability

Any REST API with an OpenAPI 3.0 spec can be called directly by an agent — no wrapper code needed.

In [ ]:
# Sample OpenAPI spec for a Threat Intelligence REST API
ti_openapi_spec = {
    'openapi': '3.0.0',
    'info': {'title': 'Threat Intel API', 'version': '1.0.0', 'description': 'Corporate threat intelligence lookup service'},
    'servers': [{'url': 'https://ti.corp.internal/api/v1'}],
    'paths': {
        '/ioc/{indicator}': {
            'get': {
                'operationId': 'lookupIOC',
                'summary': 'Look up a threat indicator (IP, domain, or hash)',
                'parameters': [{
                    'name': 'indicator',
                    'in': 'path',
                    'required': True,
                    'schema': {'type': 'string'},
                    'description': 'The IOC value to look up: IPv4 address, domain name, or file hash'
                }],
                'responses': {'200': {'description': 'Threat intel result'}}
            }
        },
        '/campaigns/active': {
            'get': {
                'operationId': 'listActiveCampaigns',
                'summary': 'List currently active threat campaigns targeting this sector',
                'responses': {'200': {'description': 'Campaign listing'}}
            }
        }
    }
}

import json
print('OpenAPI spec for Threat Intel API:')
print(json.dumps(ti_openapi_spec, indent=2))

In [ ]:
if not MOCK_MODE:
    from azure.ai.projects.models import OpenApiTool, OpenApiAnonymousAuthDetails

    # For internal APIs with managed identity: use OpenApiManagedAuthDetails with a connection
    # For public or anonymous APIs: OpenApiAnonymousAuthDetails
    auth = OpenApiAnonymousAuthDetails()

    openapi_tool = OpenApiTool(
        name='threat_intel_api',
        spec=ti_openapi_spec,
        description='Corporate threat intelligence REST API for IOC lookups and campaign tracking',
        auth=auth,
    )

    ti_agent = project_client.agents.create_version(
        agent_name='soc-openapi-ti-agent',
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions=(
                'You are a threat intelligence analyst. Use the threat intel API to look '
                'up IOCs and identify relevant active campaigns.'
            ),
            tools=[openapi_tool],
        ),
    )
    print(f'✅ OpenAPI agent created: {ti_agent.name}')
else:
    print('⚠️  Skipping OpenAPI agent creation')
    print('   To use: add your OpenAPI 3.0 spec and set auth credentials')


## 🔵 Tool Selection Decision Guide

In [ ]:
decision_data = {
    'Scenario': [
        'Simple SIEM KQL query',
        'Complex SIEM with session management',
        'External REST API (threat intel)',
        'Defender XDR — read only',
        'Defender XDR — remediation actions',
        'Multi-platform integration hub',
        'Real-time web OSINT',
    ],
    'Recommended Tool': ['FunctionTool', 'MCPTool', 'OpenApiTool', 'FunctionTool or MCPTool', 'MCPTool (approval=always)', 'MCPTool', 'BingGroundingTool'],
    'Why': [
        'Simple callable, full Python control',
        'MCP maintains session, handles auth natively',
        'No wrapper code — spec is the integration',
        'Either works; MCP if server available',
        'Explicit human-in-loop approval required',
        'MCP standardizes protocol across platforms',
        'Live web results with citation references',
    ]
}

df_decision = pd.DataFrame(decision_data)
display(df_decision.style.set_properties(**{'background-color': '#1e2127', 'color': '#e6e6e6'})
        .map(lambda v: 'color: #4ecdc4' if 'Function' in str(v) else
                       'color: #ff6b6b' if 'MCP' in str(v) else
                       'color: #45b7d1' if 'OpenApi' in str(v) else
                       'color: #ffdd44' if 'Bing' in str(v) else '', subset=['Recommended Tool']))


## 🔴 Cleanup

In [ ]:
if not MOCK_MODE:
    for agent_ref in [soc_analyst]:
        try:
            project_client.agents.delete_version(agent_name=agent_ref.name, agent_version=agent_ref.version)
            print(f'🗑️  Deleted {agent_ref.name}')
        except Exception:
            pass
    openai_client.close()
    project_client.close()
else:
    print('⚠️  Nothing to clean up in mock mode.')
